<a href="https://colab.research.google.com/github/zoesuhnny/data_science_guide/blob/main/llm_notes_from_hugging_face.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# llm notes'
import torch
from transformers import BertTokenizer, BertModel, AutoTokenizer, DataCollatorWithPadding
from datasets import load_dataset

ds = load_dataset("nyu-mll/glue", "mnli")

In [4]:
checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint) #loaded tokenizer

In [11]:
model = BertModel.from_pretrained(checkpoint) #loads model

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [12]:
model.eval()

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

In [16]:
tokens = tokenizer.tokenize("The fox swims gleefully.") #TOKENIZES sequence.
tokens

['the', 'fox', 'swim', '##s', 'glee', '##fully', '.']

In [13]:
tokenizer("The fox swims gleefully") #gives variable of input_ids, token_type_ids, and attention_mask.

{'input_ids': [101, 1996, 4419, 9880, 2015, 18874, 7699, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1]}

In [14]:
tokenizer.save_pretrained("directory_on_my_computer") #saving a tokenizer

('directory_on_my_computer/tokenizer_config.json',
 'directory_on_my_computer/tokenizer.json')

In [15]:
tokenizer = AutoTokenizer.from_pretrained(checkpoint) #instantiates (new) tokenizer for checkpoint.


In [18]:
ids = tokenizer.convert_tokens_to_ids(tokens)
ids

[1996, 4419, 9880, 2015, 18874, 7699, 1012]

In [20]:
tokenizer.convert_ids_to_tokens(ids) #gives the tokens in a list.

['the', 'fox', 'swim', '##s', 'glee', '##fully', '.']

In [21]:
tokenizer.decode(ids) #.decode() gives the original sentence.

'the fox swims gleefully.'

In [23]:
tensor_ids = torch.tensor(ids) #gives ids in tensor form, BECAUSE the model needs ids in tensor form (pytorch).
tensor_ids

tensor([ 1996,  4419,  9880,  2015, 18874,  7699,  1012])

In [26]:
tensor_ids.shape #we need this to be a 2d array, not 1d.

torch.Size([7])

In [30]:
tensor_ids_reshaped = tokenizer("The fox swims gleefully", return_tensors="pt") #return everything in tensor form.
print(tensor_ids_reshaped)
tensor_ids_reshaped = tokenizer("The fox swims gleefully", return_tensors="pt")["input_ids"] #only want input ids
print()
tensor_ids_reshaped

{'input_ids': tensor([[  101,  1996,  4419,  9880,  2015, 18874,  7699,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1]])}



tensor([[  101,  1996,  4419,  9880,  2015, 18874,  7699,   102]])

In [33]:
from transformers import AutoModelForSequenceClassification
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [34]:
model

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [37]:
model(tensor_ids_reshaped).logits

tensor([[-0.0364,  0.0900]], grad_fn=<AddmmBackward0>)